# Proyecto 3 — Credit Scoring y Segmentación de Riesgo Crediticio
## Notebook 1: Análisis Exploratorio y WoE/IV

**Objetivo:** Comprender las características del dataset crediticio, analizar el poder predictivo de cada variable mediante Information Value (IV) y preparar los datos para el modelado.

**Dataset:** Give Me Some Credit (Kaggle) — 150.000 solicitantes de crédito  
**Target:** `SeriousDlqin2yrs` — impago de ≥90 días en los próximos 2 años

---

In [ ]:
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from utils import (
    load_dataset, impute_missing, cap_outliers, create_derived_features,
    compute_woe_iv, compute_all_iv, FEATURE_COLS, TARGET
)

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.float_format', '{:.4f}'.format)
print('Setup completo.')

## 1. Carga y auditoría inicial

In [ ]:
df = load_dataset('../data/raw/cs-training.csv')
print(f'Shape: {df.shape}')
print(f'\nTasa de default: {df[TARGET].mean():.2%}')
print(f'Registros con default: {df[TARGET].sum():,}')
df.head()

In [ ]:
# Auditoría de calidad de datos
audit = pd.DataFrame({
    'dtype': df.dtypes,
    'n_missing': df.isna().sum(),
    'pct_missing': (df.isna().mean() * 100).round(2),
    'n_unique': df.nunique(),
    'min': df.min(),
    'max': df.max(),
    'mean': df.mean().round(4),
})
print('\n=== Auditoría de calidad de datos ===')
audit.style.background_gradient(subset=['pct_missing'], cmap='YlOrRd')

## 2. Distribución del target

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df[TARGET].value_counts()
axes[0].bar(['No Default (0)', 'Default (1)'], counts, color=['steelblue', 'coral'], edgecolor='white')
for i, v in enumerate(counts):
    axes[0].text(i, v + 500, f'{v:,}\n({v/len(df):.1%})', ha='center', fontsize=10)
axes[0].set_title('Distribución del Target', fontsize=12)
axes[0].set_ylabel('Número de registros')

axes[1].pie(counts, labels=['No Default', 'Default'], autopct='%1.2f%%',
            colors=['steelblue', 'coral'], startangle=90)
axes[1].set_title('Proporción de clases', fontsize=12)

plt.suptitle('Dataset altamente desbalanceado — Ratio 14:1 (No Default:Default)', fontsize=11)
plt.tight_layout()
plt.show()

## 3. Análisis univariante de features

In [ ]:
df_clean = impute_missing(df)
df_clean = cap_outliers(df_clean)

fig, axes = plt.subplots(4, 3, figsize=(18, 16))
axes = axes.ravel()

for i, col in enumerate(FEATURE_COLS):
    for label, grp in df_clean.groupby(TARGET):
        axes[i].hist(grp[col], bins=30, alpha=0.5, density=True,
                     label=f'Default={label}', edgecolor='white', linewidth=0.3)
    axes[i].set_title(col, fontsize=9, fontweight='bold')
    axes[i].set_xlabel('Valor')
    axes[i].legend(fontsize=7)

for j in range(len(FEATURE_COLS), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribución de variables por clase (Default=0 vs 1)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 4. Information Value (IV) — Poder predictivo

In [ ]:
iv_series = compute_all_iv(df_clean)

fig, ax = plt.subplots(figsize=(12, 6))
colors = ['darkred' if v > 0.3 else 'coral' if v > 0.1 else 'steelblue' if v > 0.02 else 'gray'
          for v in iv_series.values]
bars = ax.barh(iv_series.index, iv_series.values, color=colors, edgecolor='white')

ax.axvline(0.02, color='gray', linestyle='--', alpha=0.7, label='IV=0.02 (sin valor)')
ax.axvline(0.1, color='blue', linestyle='--', alpha=0.7, label='IV=0.1 (predictor débil)')
ax.axvline(0.3, color='red', linestyle='--', alpha=0.7, label='IV=0.3 (predictor fuerte)')

for bar, val in zip(bars, iv_series.values):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center', fontsize=9)

ax.set_xlabel('Information Value (IV)', fontsize=11)
ax.set_title('Ranking de Variables por Poder Predictivo (IV)', fontsize=13, fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

print('\n=== Interpretación del IV ===')
for feat, iv in iv_series.items():
    if iv < 0.02:
        strength = 'Sin valor predictivo'
    elif iv < 0.1:
        strength = 'Predictor débil'
    elif iv < 0.3:
        strength = 'Predictor medio'
    else:
        strength = 'Predictor fuerte'
    print(f'  {feat:<48} IV={iv:.3f} → {strength}')

## 5. WoE — Análisis de la variable más predictiva

In [ ]:
top_feature = iv_series.index[0]
woe_table, iv = compute_woe_iv(df_clean, top_feature)

print(f'Feature: {top_feature} | IV: {iv:.4f}')
print(woe_table[['bin', 'events', 'non_events', 'woe', 'iv_contribution']].to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(len(woe_table)), woe_table['woe'],
            color=['coral' if v > 0 else 'steelblue' for v in woe_table['woe']],
            edgecolor='white')
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_title(f'Weight of Evidence — {top_feature}', fontsize=11)
axes[0].set_xlabel('Bin')
axes[0].set_ylabel('WoE')

axes[1].plot(range(len(woe_table)), woe_table['events'] / (woe_table['events'] + woe_table['non_events']),
             marker='o', color='coral', linewidth=2)
axes[1].set_title(f'Tasa de Default por Bin — {top_feature}', fontsize=11)
axes[1].set_xlabel('Bin')
axes[1].set_ylabel('Tasa de Default')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.1%}'))

plt.tight_layout()
plt.show()

## 6. Correlaciones y multicolinealidad

In [ ]:
df_eng = create_derived_features(df_clean)
all_features = FEATURE_COLS + ['total_late_payments', 'high_utilization', 'debt_per_dependent']

corr = df_eng[all_features].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5, square=False, ax=ax,
            annot_kws={'size': 7})
ax.set_title('Matriz de Correlación — Features Crediticias', fontsize=13)
plt.tight_layout()
plt.show()

## 7. Resumen y próximos pasos

### Hallazgos clave

| Variable | IV | Hallazgo |
|----------|----|---------|
| `RevolvingUtilizationOfUnsecuredLines` | > 0.3 | Predictor fuerte — alta utilización ↑ riesgo |
| `NumberOfTimes90DaysLate` | > 0.3 | Predictor fuerte — mora grave pasada ↑ riesgo |
| `age` | 0.1-0.3 | Predictor medio — clientes jóvenes tienen mayor riesgo |
| `DebtRatio` | < 0.1 | Débil solo — necesita interacción con ingresos |

### Decisiones de preprocesamiento
1. **Imputar** `MonthlyIncome` (18.5% missing) y `NumberOfDependents` (2.6%) con mediana
2. **Capear** outliers extremos (percentil 99) en variables de utilización y mora
3. **Crear** features derivadas: `total_late_payments`, `high_utilization`, `debt_per_dependent`
4. **Aplicar** SMOTE solo sobre training para compensar desbalance

**Próximo notebook:** `02_modeling_scoring.ipynb` — XGBoost, LightGBM y Scorecard